In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, HBox, VBox, HTML, Layout
from IPython.display import display

# ============================================================
# TRANSPOSITION THEOREM — EQUIVALENT DIGITAL FILTER STRUCTURES
# ============================================================

plt.ioff()

CONTENT_WIDTH = '900px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12,'axes.labelsize':10.5,'xtick.labelsize':9,'ytick.labelsize':9,'legend.fontsize':8.5})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.tr-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.tr-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:10px 14px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.tr-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:9px 13px;
    border-radius:0 0 8px 8px;
    font-size:14px;
    line-height:1.48;
    margin-bottom:7px;
}

.tr-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:8px 11px;
    margin-bottom:7px;
    font-size:13.5px;
    line-height:1.48;
}

.tr-result{
    background:#fff8e6;
    border:1px solid #d8b451;
}

.tr-title{
    color:#0d47a1;
    font-weight:bold;
    font-size:14.5px;
    margin-bottom:5px;
}

.tr-equation{
    text-align:center;
    font-family:serif;
    font-size:16px;
    margin:6px 0;
}

.tr-cols{
    display:flex;
    gap:12px;
}

.tr-col{
    flex:1;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="tr-root">

<div class="tr-header">
Transposition Theorem — Equivalent Digital Filter Structures
</div>

<div class="tr-doc">

According to the transposition theorem, reversing the directions of all
branches, interchanging branching and summation operations, and exchanging
the input and output produces a structure with the <b>same transfer function</b>
as the original signal-flow graph.

The second-order recursive system used in this notebook has transfer function

<div class="tr-equation">
<b>
H(z) =
(β₀ + β₁z⁻¹ + β₂z⁻²) /
(1 − α₁z⁻¹ − α₂z⁻²).
</b>
</div>

The purpose of the notebook is to implement the original and transposed
structures <b>independently</b>, apply exactly the same input signal to both,
and verify that their output sequences are identical apart from numerical
roundoff.

<div style="text-align:center;font-size:14.5px;margin:7px 0;">
<b>
Different internal structures → Same transfer function → Same output
</b>
</div>

The coefficient controls modify both realizations simultaneously. The
signal-flow graphs themselves remain fixed; only their coefficient values
and the resulting output sequences are updated.

</div>

</div>
"""))

# ============================================================
# CONTROLS
# ============================================================

alpha1_slider = FloatSlider(value=1.20,min=-1.50,max=1.50,step=0.05,description='α₁:',continuous_update=True,readout_format='.2f',style={'description_width':'25px'},layout=Layout(width='170px'))

alpha2_slider = FloatSlider(value=-0.45,min=-0.90,max=0.90,step=0.05,description='α₂:',continuous_update=True,readout_format='.2f',style={'description_width':'25px'},layout=Layout(width='170px'))

beta0_slider = FloatSlider(value=0.30,min=-1.50,max=1.50,step=0.05,description='β₀:',continuous_update=True,readout_format='.2f',style={'description_width':'25px'},layout=Layout(width='170px'))

beta1_slider = FloatSlider(value=0.20,min=-1.50,max=1.50,step=0.05,description='β₁:',continuous_update=True,readout_format='.2f',style={'description_width':'25px'},layout=Layout(width='170px'))

beta2_slider = FloatSlider(value=0.10,min=-1.50,max=1.50,step=0.05,description='β₂:',continuous_update=True,readout_format='.2f',style={'description_width':'25px'},layout=Layout(width='170px'))

controls = HBox([alpha1_slider,alpha2_slider,beta0_slider,beta1_slider,beta2_slider],layout=Layout(width=CONTENT_WIDTH,border='1px solid #b9cce5',padding='7px 8px',margin='0 0 2px 0'))

# ============================================================
# EQUATIONS OF THE TWO REALIZATIONS
# ============================================================

display(HTML("""
<div class="tr-root">

<div class="tr-cols">

<div class="tr-col">

<div class="tr-box">

<div class="tr-title">
Original realization
</div>

<div class="tr-equation">
y[n] = α₁y[n−1] + α₂y[n−2]
</div>

<div class="tr-equation">
+ β₀x[n] + β₁x[n−1] + β₂x[n−2]
</div>

This is the ordinary second-order difference equation associated with the
transfer function.

</div>

</div>

<div class="tr-col">

<div class="tr-box">

<div class="tr-title">
Transposed realization
</div>

<div class="tr-equation">
y[n] = β₀x[n] + v₁[n−1]
</div>

<div class="tr-equation">
v₁[n] = α₁y[n] + β₁x[n] + v₂[n−1]
</div>

<div class="tr-equation">
v₂[n] = α₂y[n] + β₂x[n]
</div>

The internal state variables are different, but the input-output behavior is
unchanged.

</div>

</div>

</div>

</div>
"""))

# ============================================================
# DIRECT REALIZATION
# ============================================================

def direct_realization(x,alpha1,alpha2,beta0,beta1,beta2):

    N = len(x)

    y = np.zeros(N)

    for n in range(N):

        xn = x[n]

        x1 = x[n-1] if n >= 1 else 0.0
        x2 = x[n-2] if n >= 2 else 0.0

        y1 = y[n-1] if n >= 1 else 0.0
        y2 = y[n-2] if n >= 2 else 0.0

        y[n] = alpha1*y1+alpha2*y2+beta0*xn+beta1*x1+beta2*x2

    return y

# ============================================================
# TRANSPOSED REALIZATION
# ============================================================

def transposed_realization(x,alpha1,alpha2,beta0,beta1,beta2):

    N = len(x)

    y = np.zeros(N)

    v1_delay = 0.0
    v2_delay = 0.0

    for n in range(N):

        y[n] = beta0*x[n]+v1_delay

        v1 = alpha1*y[n]+beta1*x[n]+v2_delay

        v2 = alpha2*y[n]+beta2*x[n]

        v1_delay = v1
        v2_delay = v2

    return y

# ============================================================
# TEST INPUT — CREATED ONCE
# ============================================================

N = 60

n = np.arange(N)

x = np.zeros(N)

x[0] = 1.0

x += 0.25*np.sin(0.18*np.pi*n)

# ============================================================
# INITIAL VALUES
# ============================================================

alpha1 = alpha1_slider.value
alpha2 = alpha2_slider.value
beta0 = beta0_slider.value
beta1 = beta1_slider.value
beta2 = beta2_slider.value

y_direct = direct_realization(x,alpha1,alpha2,beta0,beta1,beta2)

y_transposed = transposed_realization(x,alpha1,alpha2,beta0,beta1,beta2)

difference = y_direct-y_transposed

# ============================================================
# NUMERICAL VERIFICATION BOX
# ============================================================

result_html = HTML(layout=Layout(width=CONTENT_WIDTH))

# ============================================================
# FIGURE 1 — SIGNAL-FLOW GRAPHS
# CREATED ONCE
# ============================================================

fig1,(ax_original,ax_transposed) = plt.subplots(1,2,figsize=(9.0,3.6))

fig1.canvas.toolbar_visible = False
fig1.canvas.header_visible = False
fig1.canvas.footer_visible = False

# ============================================================
# ORIGINAL STRUCTURE
# ============================================================

ax_original.set_xlim(-0.8,6.7)

ax_original.set_ylim(-2.25,1.7)

ax_original.axis('off')

ax_original.set_title('Original Structure')

original_nodes = {
    'sum':(1.55,0.55),
    'mid':(3.45,-0.20),
    'bot':(3.45,-1.45)
}

for x0,y0 in original_nodes.values():

    ax_original.add_patch(plt.Circle((x0,y0),0.105,fill=False,linewidth=1.2))

arrow = {'arrowstyle':'->','linewidth':1.3,'shrinkA':4,'shrinkB':4}

ax_original.annotate('',xy=original_nodes['sum'],xytext=(-0.25,0.55),arrowprops=arrow)

ax_original.annotate('',xy=(6.10,0.55),xytext=original_nodes['sum'],arrowprops=arrow)

ax_original.text(-0.48,0.68,r'$x[n]$',ha='center',va='center',fontsize=11,fontweight='bold')

ax_original.text(6.34,0.68,r'$y[n]$',ha='center',va='center',fontsize=11,fontweight='bold')

original_beta0_text = ax_original.text(0.72,0.93,rf'$\beta_0={beta0:.2f}$',ha='center',fontsize=10)

ax_original.annotate('',xy=original_nodes['bot'],xytext=original_nodes['mid'],arrowprops=arrow)

ax_original.text(3.62,-0.86,r'$z^{-1}$',fontsize=10)

ax_original.annotate('',xy=original_nodes['mid'],xytext=(5.00,0.52),arrowprops=arrow)

original_beta1_text = ax_original.text(4.82,-0.05,rf'$\beta_1={beta1:.2f}$',ha='left',fontsize=10)

feedback1 = {'arrowstyle':'->','linewidth':1.3,'connectionstyle':'arc3,rad=-0.35','shrinkA':4,'shrinkB':4}

feedback2 = {'arrowstyle':'->','linewidth':1.3,'connectionstyle':'arc3,rad=-0.52','shrinkA':4,'shrinkB':4}

ax_original.annotate('',xy=original_nodes['sum'],xytext=original_nodes['mid'],arrowprops=feedback1)

ax_original.annotate('',xy=original_nodes['sum'],xytext=original_nodes['bot'],arrowprops=feedback2)

original_alpha1_text = ax_original.text(0.92,-0.12,rf'$\alpha_1={alpha1:.2f}$',ha='center',fontsize=10)

original_alpha2_text = ax_original.text(1.30,-1.48,rf'$\alpha_2={alpha2:.2f}$',ha='center',fontsize=10)

original_beta2_text = ax_original.text(4.72,-1.38,rf'$\beta_2={beta2:.2f}$',ha='left',fontsize=10)

# ============================================================
# TRANSPOSED STRUCTURE
# ============================================================

ax_transposed.set_xlim(-0.9,6.9)

ax_transposed.set_ylim(-2.35,1.7)

ax_transposed.axis('off')

ax_transposed.set_title('Transposed Structure')

transposed_nodes = {
    'left':(1.55,0.55),
    'top':(3.55,0.55),
    'bot':(5.10,-1.00)
}

for x0,y0 in transposed_nodes.values():

    ax_transposed.add_patch(plt.Circle((x0,y0),0.105,fill=False,linewidth=1.2))

ax_transposed.annotate('',xy=transposed_nodes['left'],xytext=(-0.20,0.55),arrowprops=arrow)

ax_transposed.annotate('',xy=(6.45,0.55),xytext=transposed_nodes['top'],arrowprops=arrow)

ax_transposed.text(-0.45,0.68,r'$x[n]$',ha='center',va='center',fontsize=11,fontweight='bold')

ax_transposed.text(6.65,0.68,r'$y[n]$',ha='center',va='center',fontsize=11,fontweight='bold')

transposed_beta0_text = ax_transposed.text(0.82,0.78,rf'$\beta_0={beta0:.2f}$',ha='center',fontsize=10)

# ------------------------------------------------------------
# TOP HORIZONTAL DELAY
# ------------------------------------------------------------

ax_transposed.annotate('',xy=transposed_nodes['top'],xytext=transposed_nodes['left'],arrowprops=arrow)

ax_transposed.text(1.85,0.92,r'$z^{-1}$',ha='center',fontsize=10)

# ------------------------------------------------------------
# DIAGONAL DELAY
# ------------------------------------------------------------

ax_transposed.annotate('',xy=transposed_nodes['bot'],xytext=transposed_nodes['top'],arrowprops=arrow)

ax_transposed.text(3.55,-0.08,r'$z^{-1}$',ha='right',va='center',fontsize=10)

# ------------------------------------------------------------
# BETA 1 BRANCH
# ------------------------------------------------------------

ax_transposed.annotate('',xy=transposed_nodes['top'],xytext=(0.20,-0.75),arrowprops=arrow)

transposed_beta1_text = ax_transposed.text(1.55,-0.82,rf'$\beta_1={beta1:.2f}$',ha='center',fontsize=10)

# ------------------------------------------------------------
# BETA 2 BRANCH
# ------------------------------------------------------------

ax_transposed.annotate('',xy=transposed_nodes['bot'],xytext=(0.20,-1.55),arrowprops=arrow)

transposed_beta2_text = ax_transposed.text(1.70,-1.72,rf'$\beta_2={beta2:.2f}$',ha='center',fontsize=10)

# ------------------------------------------------------------
# ALPHA 1 BRANCH
# ------------------------------------------------------------

ax_transposed.annotate('',xy=transposed_nodes['top'],xytext=(6.15,-0.55),arrowprops=arrow)

transposed_alpha1_text = ax_transposed.text(3.80,-0.42,rf'$\alpha_1={alpha1:.2f}$',ha='center',va='center',fontsize=10)

# ------------------------------------------------------------
# ALPHA 2 BRANCH
# ------------------------------------------------------------

ax_transposed.annotate('',xy=transposed_nodes['bot'],xytext=(6.15,-1.65),arrowprops=arrow)

transposed_alpha2_text = ax_transposed.text(5.70,-1.88,rf'$\alpha_2={alpha2:.2f}$',ha='center',fontsize=10)

plt.subplots_adjust(left=0.04,right=0.98,top=0.86,bottom=0.08,wspace=0.18)

# ============================================================
# FIGURE 2 — OUTPUT COMPARISON
# CREATED ONCE
# ============================================================

fig2,(ax_output,ax_difference) = plt.subplots(1,2,figsize=(9.0,3.35))

fig2.canvas.toolbar_visible = False
fig2.canvas.header_visible = False
fig2.canvas.footer_visible = False

# ============================================================
# OUTPUT CURVES
# ============================================================

direct_line, = ax_output.plot(n,y_direct,linewidth=1.4,label='Original structure')

transposed_line, = ax_output.plot(n,y_transposed,'--',linewidth=1.2,label='Transposed structure')

ax_output.set_xlim(0,N-1)

ax_output.set_title('Output Sequences')

ax_output.set_xlabel('Sample index n')

ax_output.set_ylabel('y[n]')

ax_output.grid(True,linestyle=':',alpha=0.25)

ax_output.legend(loc='upper center',bbox_to_anchor=(0.5,-0.17),ncol=2,frameon=False)

# ============================================================
# DIFFERENCE CURVE
# ============================================================

difference_line, = ax_difference.plot(n,difference,'ro-',linewidth=0.8,markersize=3)

ax_difference.axhline(0,linewidth=0.8)

ax_difference.set_xlim(0,N-1)

ax_difference.set_title('Difference Between the Outputs')

ax_difference.set_xlabel('Sample index n')

ax_difference.set_ylabel(r'$y_D[n]-y_T[n]$')

ax_difference.grid(True,linestyle=':',alpha=0.25)

# ============================================================
# INITIAL OUTPUT SCALES
# ============================================================

output_limit = max(1.0,1.10*np.max(np.abs(np.concatenate((y_direct,y_transposed)))))

difference_limit = max(1e-15,1.10*np.max(np.abs(difference)))

ax_output.set_ylim(-output_limit,output_limit)

ax_difference.set_ylim(-difference_limit,difference_limit)

plt.subplots_adjust(left=0.08,right=0.98,top=0.88,bottom=0.22,wspace=0.28)

# ============================================================
# UPDATE CALLBACK
# ============================================================

def update(change=None):

    alpha1 = alpha1_slider.value
    alpha2 = alpha2_slider.value
    beta0 = beta0_slider.value
    beta1 = beta1_slider.value
    beta2 = beta2_slider.value

    # --------------------------------------------------------
    # UPDATE BOTH REALIZATIONS
    # --------------------------------------------------------

    y_direct = direct_realization(x,alpha1,alpha2,beta0,beta1,beta2)

    y_transposed = transposed_realization(x,alpha1,alpha2,beta0,beta1,beta2)

    difference = y_direct-y_transposed

    # --------------------------------------------------------
    # UPDATE SIGNAL-FLOW GRAPH LABELS ONLY
    # --------------------------------------------------------

    original_beta0_text.set_text(rf'$\beta_0={beta0:.2f}$')

    original_beta1_text.set_text(rf'$\beta_1={beta1:.2f}$')

    original_beta2_text.set_text(rf'$\beta_2={beta2:.2f}$')

    original_alpha1_text.set_text(rf'$\alpha_1={alpha1:.2f}$')

    original_alpha2_text.set_text(rf'$\alpha_2={alpha2:.2f}$')

    transposed_beta0_text.set_text(rf'$\beta_0={beta0:.2f}$')

    transposed_beta1_text.set_text(rf'$\beta_1={beta1:.2f}$')

    transposed_beta2_text.set_text(rf'$\beta_2={beta2:.2f}$')

    transposed_alpha1_text.set_text(rf'$\alpha_1={alpha1:.2f}$')

    transposed_alpha2_text.set_text(rf'$\alpha_2={alpha2:.2f}$')

    # --------------------------------------------------------
    # UPDATE OUTPUT CURVES ONLY
    # --------------------------------------------------------

    direct_line.set_ydata(y_direct)

    transposed_line.set_ydata(y_transposed)

    difference_line.set_ydata(difference)

    # --------------------------------------------------------
    # UPDATE ONLY THE Y-AXIS LIMITS
    # --------------------------------------------------------

    output_limit = max(1.0,1.10*np.max(np.abs(np.concatenate((y_direct,y_transposed)))))

    difference_limit = max(1e-15,1.10*np.max(np.abs(difference)))

    ax_output.set_ylim(-output_limit,output_limit)

    ax_difference.set_ylim(-difference_limit,difference_limit)

    # --------------------------------------------------------
    # NUMERICAL VERIFICATION
    # --------------------------------------------------------

    poles = np.roots([1.0,-alpha1,-alpha2])

    maximum_pole_radius = np.max(np.abs(poles))

    stable = maximum_pole_radius < 1.0

    maximum_difference = np.max(np.abs(difference))

    status = "STABLE" if stable else "UNSTABLE"

    result_html.value = f"""
    <div class="tr-root">

    <div class="tr-box tr-result">

    <div class="tr-title">
    Numerical verification
    </div>

    <div class="tr-equation">
    H(z) =
    ({beta0:.2f} {beta1:+.2f}z⁻¹ {beta2:+.2f}z⁻²) /
    (1 {(-alpha1):+.2f}z⁻¹ {(-alpha2):+.2f}z⁻²)
    </div>

    Maximum pole radius:
    <b>{maximum_pole_radius:.6f}</b>

    &nbsp;&nbsp;&nbsp;

    System:
    <b>{status}</b>

    <br>

    <div class="tr-equation">
    <b>
    max |y<sub>direct</sub>[n] − y<sub>transposed</sub>[n]|
    = {maximum_difference:.3e}
    </b>
    </div>

    A value near machine precision confirms the numerical equivalence of the
    two structures.

    </div>

    </div>
    """

    # --------------------------------------------------------
    # REDRAW EXISTING CANVASES ONLY
    # --------------------------------------------------------

    fig1.canvas.draw_idle()

    fig2.canvas.draw_idle()

# ============================================================
# OBSERVERS
# ============================================================

alpha1_slider.observe(update,names='value')

alpha2_slider.observe(update,names='value')

beta0_slider.observe(update,names='value')

beta1_slider.observe(update,names='value')

beta2_slider.observe(update,names='value')

# ============================================================
# DISPLAY
# ============================================================

display(result_html)

display(fig1.canvas)

display(controls)

display(fig2.canvas)

# ============================================================
# INITIAL UPDATE
# ============================================================

update()